# Text II: Searching for keywords in UN speeches

Last week we worked with strings, loops and files, and found the longest word in a text. Today we use the same tools on **real political texts**: speeches that heads of state give every September at the **UN General Debate**.

Our question for today:

> **When countries talk about *energy*, which words do they put next to it?**
> Do they say *clean* energy, *nuclear* energy, *renewable* energy...?

Goal: **search for a keyword**, look at the **words around it**, and find **sub-topics** like *clean energy* or *nuclear energy*.

### Our plan

```
 table of 200 speeches
        │
        ▼
 1. take ONE speech            ──►  a long string
        │
        ▼
 2. clean it                   ──►  lower case, a list of words
        │
        ▼
 3. find the keyword           ──►  how often does "energy" appear?
        │
        ▼
 4. look at its neighbours     ──►  the word before, the word after
        │
        ▼
 5. count the neighbours       ──►  "renewable" 3x, "nuclear" 2x ...
        │
        ▼
 6. do it for ALL speeches     ──►  a table of hits, saved as a CSV
```

**Why the table at the end matters:** next week we will turn it into **charts with seaborn**, and in two weeks we will send each hit to a **large language model**. So today we build the material for the next two lectures.

> **About the data**
>
> The speeches come from the *United Nations General Debate Corpus* (UNGDC), which is free to use (CC0 licence). We took **20 countries** and the years **2015 to 2024**: 200 speeches in total, all in English.
>
> Source: Jankin, S., Baturo, A., & Dasandi, N. (2025). *Words to unite nations: The complete United Nations General Debate Corpus, 1946–present.* Journal of Peace Research, 62(4), 1339–1351.

### How to use this notebook

- **Predict** means: before you run the cell, say (or write down) what you think it will print. Being wrong is fine; that is how you learn what the code really does.
- Cells with a `# ...` comment are for **you** to write. If you are stuck, open the *Show solution* box below it.
- Run a cell with **Shift + Enter**.

## 0. Open our tools

`pandas` reads tables. `re` is Python's module for searching in text; we will use it once, for one cleaning step.

In [ ]:
import pandas as pd
import re

## 1. Load the speeches

The file `ungd_speeches.csv` is in the **same folder as this notebook**. That is why the file name alone is enough: Python looks for it next to the notebook.

In [ ]:
speeches = pd.read_csv("ungd_speeches.csv")

<details>
<summary>Got a <code>FileNotFoundError</code>? Click here.</summary>

The notebook and the CSV are not in the same folder (for example, you downloaded only the notebook). No problem: read the file directly from GitHub instead.

```python
url = "https://raw.githubusercontent.com/dsl-unibe-ch/wti_python_course/main/Lecture_7/ungd_speeches.csv"
speeches = pd.read_csv(url)
```

</details>

Let's look at the table. **Predict:** how many rows and how many columns will it have?

In [ ]:
speeches.head()

In [ ]:
speeches.shape

**Checkpoint:** 200 rows (one per speech) and 4 columns: `country`, `country_name`, `year`, and `text`.

The `text` column contains the **whole speech**, often more than 2,000 words. pandas shows only the beginning of it.

## 2. Take ONE speech

We learn everything on one speech first. Only at the very end do we repeat it for all 200.

We take the row with **index 2**: Brazil, 2015. `speeches.loc[2, "text"]` means *row 2, column "text"*.

In [ ]:
print(speeches.loc[2, "country_name"], speeches.loc[2, "year"])
text = speeches.loc[2, "text"]

**Predict:** what kind of value is `text`? A number, a list, a string...?

In [ ]:
# Print the type of text, its length with len(), and its first 500 characters (text[:500])


<details>
<summary>Show solution</summary>

```python
print(type(text))
print(len(text))
print(text[:500])
```

`text` is one long **string**. `len()` counts characters, not words. `text[:500]` is slicing from last week: characters 0 to 499.

</details>

## 3. Clean the speech

### 3a. Everything to lower case

For Python, `Energy` and `energy` are **different words**. To catch both, we first turn the whole speech into lower case.

In [ ]:
# Make a new variable text_lower with the lower-case speech, then print its first 300 characters


<details>
<summary>Show solution</summary>

```python
text_lower = text.lower()
print(text_lower[:300])
```

Remember from last week: string methods give back a **new** string. We must store it with `=`, otherwise the result is lost.

</details>

### 3b. Cut the speech into words

`split()` cuts a string at every space and gives back a **list** of words.

**Predict:** look at the first 20 words. Is anything still wrong with them?

In [ ]:
# Split text_lower into a list called words, then print the first 20 words: words[:20]


<details>
<summary>Show solution</summary>

```python
words = text_lower.split()
print(words[:20])
```

Look closely: some words still have **punctuation** attached, like `"birthday."` at position 18. For Python, `energy.` and `energy` are different words. It is the same problem we had with `(hereinafter,` last week.

</details>

### 3c. A ready-made recipe to remove punctuation

Instead of `split()`, we use one line with `re.findall()`. You do not need to understand the pattern today, only what it does:

> `re.findall(r"[a-z]+", text_lower)` means **"give me every group of letters a to z, as a list"**.

Everything that is not a letter (spaces, dots, commas, brackets, numbers) is thrown away.

In [ ]:
words = re.findall(r"[a-z]+", text_lower)
print(words[:20])
print(len(words))

**Checkpoint:** the words are clean now, and `len(words)` tells us how many words the speech has (about 2,500).

## 4. Find the keyword

We store the keyword in a variable. At the end of the notebook you only have to change **this one line** to search for something else.

In [ ]:
keyword = "energy"

### 4a. The quick way: `count()`

Lists have a `count()` method, just like strings.

In [ ]:
# How many times does keyword appear in words?


<details>
<summary>Show solution</summary>

```python
words.count(keyword)
```

Brazil 2015 says "energy" **10 times**.

</details>

### 4b. The same count, with a loop

`count()` gives us the answer, but it hides **how** the counting happens. We now do the same thing ourselves with a `for` loop. This is the most important idea of today, because every following step uses it.

A loop that collects something always has the same three parts:

| Step | In plain words | In Python |
|---|---|---|
| 1. **Create** (before the loop) | "I start with zero" | `count = 0` |
| 2. **Update** (inside the loop) | "for each word: if it is my keyword, add one" | `count = count + 1` |
| 3. **Use** (after the loop) | "show me the result" | `print(count)` |

First, let's just **watch** a loop walk through the first 10 words. `for word in words[:10]:` means *"take the words one by one, and each time call the current one `word`"*.

In [ ]:
for word in words[:10]:
    print(word)

Now we add a question inside the loop. **Predict:** which lines will print `True`?

In [ ]:
for word in words[:10]:
    print(word, word == "the")

Now the full counting loop, over **all** the words. Write it step by step: create, update, use.

In [ ]:
# 1. create a variable count that starts at 0
# 2. loop over all words; if the word is equal to keyword, add 1 to count
# 3. after the loop, print count


<details>
<summary>Show solution</summary>

```python
count = 0
for word in words:
    if word == keyword:
        count = count + 1
print(count)
```

Watch the **indentation**: `if` is inside the loop (4 spaces), `count = count + 1` is inside the `if` (8 spaces), and `print(count)` is **not** indented, so it runs only once, after the loop.

**Checkpoint:** the result must be the same as `words.count(keyword)`.

</details>

## 5. Look at the neighbours

To find the word **before** "energy", we need to know **where** in the list "energy" is. Every item in a list has a **position** (index), starting at 0.

In [ ]:
print(words[0])
print(words[1])
print(words[2])

If a word is at position `i`, then:

```
words[i-1]   the word before
words[i]     the word itself
words[i+1]   the word after
```

**Predict:** what will this print?

In [ ]:
i = 5
print(words[i-1], "|", words[i], "|", words[i+1])

### 5a. A loop that knows the position: `enumerate()`

`for word in words` gives us only the word. `for i, word in enumerate(words)` gives us **two things each time**: the position `i` **and** the word. Let's watch it on the first 10 words.

In [ ]:
for i, word in enumerate(words[:10]):
    print(i, word)

### 5b. Print every keyword with its neighbours

Same loop as in 4b: go through the words, check if it is the keyword. But instead of adding 1, we print the word before, the keyword, and the word after.

In [ ]:
# Loop over enumerate(words).
# If word is equal to keyword, print words[i-1], word, words[i+1]


<details>
<summary>Show solution</summary>

```python
for i, word in enumerate(words):
    if word == keyword:
        print(words[i-1], word, words[i+1])
```

This is the "sub-topic": *renewable* energy, *nuclear* energy, *electrical* energy...

</details>

## 6. Count the neighbours

Reading the lines works for one speech, but not for 200. We want Python to **count** the words before the keyword. Again: create, update, use. This time we collect words in a **list**, using `.append()`.

In [ ]:
# 1. create an empty list called before
# 2. loop over enumerate(words); if word is the keyword, append words[i-1] to before
# 3. after the loop, print before


<details>
<summary>Show solution</summary>

```python
before = []
for i, word in enumerate(words):
    if word == keyword:
        before.append(words[i-1])
print(before)
```

Same three steps as the counter in 4b. Only the "update" is different: `before.append(...)` instead of `count = count + 1`.

</details>

pandas can count a list for us. `pd.Series(before)` turns the list into a pandas column; `.value_counts()` counts how often each word appears, from most to least common.

In [ ]:
pd.Series(before).value_counts()

### 6a. Stop words

Some of the most common neighbours are words like *our*, *in*, *and*. They appear next to almost everything and tell us nothing about energy. Such words are called **stop words**. We write our own small list of them and skip them.

**Predict:** what will the two lines below print?

In [ ]:
stopwords = ["the", "and", "of", "our", "in", "to", "a", "an", "for", "on", "its",
             "their", "this", "that", "with", "as", "is", "be", "by", "s", "all", "we"]

print("our" in stopwords)
print("renewable" in stopwords)

In [ ]:
# Same loop as before, but only append words[i-1] if it is NOT in stopwords.
# Then count the result with pd.Series(...).value_counts()


<details>
<summary>Show solution</summary>

```python
before = []
for i, word in enumerate(words):
    if word == keyword:
        if words[i-1] not in stopwords:
            before.append(words[i-1])
pd.Series(before).value_counts()
```

An `if` inside an `if`: the second question is only asked when the first answer was yes.

</details>

### 6b. More context: a window of words

One word before is sometimes not enough. With **slicing** we can take several words on each side. `words[i-10:i+11]` means *from 10 words before to 10 words after*. `" ".join(...)` glues the list back into one string (the opposite of `split()`).

First, on one position, to see what the slice does:

In [ ]:
i = 784
print(words[i])
print(words[i-10:i+11])
print(" ".join(words[i-10:i+11]))

Now the same loop as before (yes, again!), but this time we print the context of **every** hit.

In [ ]:
# Loop over enumerate(words); if word is the keyword, print " ".join(words[i-10:i+11])


<details>
<summary>Show solution</summary>

```python
for i, word in enumerate(words):
    if word == keyword:
        print(" ".join(words[i-10:i+11]))
```

Same loop, same `if`. Only the line inside changed. By now you have seen this pattern five times: that is the whole secret of today's lecture.

</details>

This little piece of text around the keyword is called the **context**. In two weeks, we will give exactly these snippets to a language model and ask it questions about them.

## 7. All speeches at once

Everything so far worked on **one** speech. Now we repeat it for all 200. To do this we put our loop **inside another loop**:

```
for each speech in the table:          ◄── outer loop (200 times)
    clean the text, make the words
    for each word in this speech:      ◄── inner loop (thousands of times)
        if it is the keyword:
            write down one row: country, year, word before, word after, context
```

Each "row" we write down is a **dictionary** (from Lecture 2: `{"key": value}`), and we collect all of them in a list called `hits`. `speeches.iterrows()` gives us the table **one row at a time**, so `row["text"]` is the speech and `row["country"]` is the country.

Read the cell below carefully before running it. The inner part is **the same code you already wrote**, only with more information in each row.

There are two new **safety lines**. With 200 speeches, sooner or later a keyword is the **very last word** of a speech (then `words[i+1]` does not exist and Python stops with `IndexError: list index out of range`), or it is among the **first 10 words** (then `i-10` is negative and the slice comes back empty). The two lines prevent both problems.

In [ ]:
hits = []                                      # create (before the loops)

for index, row in speeches.iterrows():         # outer loop: one speech at a time
    words = re.findall(r"[a-z]+", row["text"].lower())
    words.append("")                           # safety: an empty word at the end, so words[i+1] always exists

    for i, word in enumerate(words):           # inner loop: one word at a time
        if word == keyword:
            start = max(0, i-10)               # safety: never start before position 0
            hits.append({                      # update: one new row per hit
                "country": row["country"],
                "country_name": row["country_name"],
                "year": row["year"],
                "keyword": keyword,
                "word_before": words[i-1],
                "word_after": words[i+1],
                "context": " ".join(words[start:i+11]),
            })

print(len(hits))                               # use (after the loops)

**Checkpoint:** you should find **247** hits for "energy" in the 200 speeches.

A list of dictionaries turns directly into a pandas table.

In [ ]:
hits_table = pd.DataFrame(hits)
hits_table.head(10)

### 7a. What does the whole table tell us?

**Predict:** which word do you think comes before *energy* most often, in all 200 speeches?

In [ ]:
# Count the values in the word_before column (use .value_counts() and show the top 15 with .head(15))


<details>
<summary>Show solution</summary>

```python
hits_table["word_before"].value_counts().head(15)
```

Stop words are back at the top, because the big loop does not skip them. That's on purpose: we keep **every** hit in the table, and decide later (next week) what to filter.

</details>

In [ ]:
# Which countries talk about the keyword the most? Count the values in the country_name column


<details>
<summary>Show solution</summary>

```python
hits_table["country_name"].value_counts()
```

This is a first comparison between countries. Next week we turn tables like this one into charts.

</details>

### 7b. Save the table

We save the hits as a CSV file, in the same folder as this notebook. The name is made from the keyword, so `energy` gives `energy_hits.csv`. **Next week, the seaborn lecture starts from this file.**

In [ ]:
hits_table.to_csv(keyword + "_hits.csv", index=False)

## 8. (Optional) Paragraphs instead of words

Each speech in our file has **one paragraph per line**. A line break is written `\n` in Python, so `split("\n")` cuts a speech into paragraphs. Then we can print every **paragraph** that talks about our keyword: much more context than 10 words.

In [ ]:
# Split text (the Brazil speech) into a list called paragraphs, using split("\n").
# Print how many paragraphs there are.
# Then loop over the paragraphs and print only those where keyword appears in paragraph.lower()


<details>
<summary>Show solution</summary>

```python
paragraphs = text.split("\n")
print(len(paragraphs))

for paragraph in paragraphs:
    if keyword in paragraph.lower():
        print(paragraph)
        print("---")
```

`keyword in paragraph.lower()` is the `in` check from last week: it answers True or False.

</details>

## 9. Your turn

Go back to **section 4**, change the keyword, and run all the cells again from there (**Run → Run Selected Cell and All Below**). Some keywords that give interesting results in these speeches:

| keyword | what to look at |
|---|---|
| `trade` | *free* trade? *world* trade? *arms* trade? |
| `sanctions` | which adjectives do countries use? Who talks about them most? |
| `debt` | *external*, *sovereign*, *unsustainable*... which countries? |
| `war` | *cold* war, *world* war, *civil* war |
| `climate` | try `word_after` instead of `word_before` |

Questions to answer for your keyword:

1. How many hits are there in all 200 speeches?
2. What are the most common **words before** and **words after** it (ignoring stop words)?
3. Which three countries use it the most?
4. Read 3 rows of the `context` column. Do the countries use the word in the same way?

**Tip:** the keyword must be **one word in lower case**, because our words are all lower case.

| Python says | What it usually means |
|---|---|
| `KeyError: 'word_before'` | Your keyword was found **0 times**, so the table is empty. Check the spelling and use lower case. |
| `NameError: name 'words' is not defined` | You skipped a cell. Run the notebook again from the top (**Run → Run All Cells**). |
| `IndentationError` | A line inside a `for` or `if` needs 4 more spaces. |
| `FileNotFoundError` | See the box in section 1: read the CSV from the GitHub link instead. |

## What we did today

| Step | Code |
|---|---|
| Load a table | `pd.read_csv("ungd_speeches.csv")` |
| Take one cell of a table | `speeches.loc[2, "text"]` |
| Lower case | `text.lower()` |
| Words without punctuation | `re.findall(r"[a-z]+", text_lower)` |
| Count one word | `words.count(keyword)` |
| Loop with a counter | `count = 0` → `count = count + 1` → `print(count)` |
| Loop with position | `for i, word in enumerate(words):` |
| Word before / after | `words[i-1]`, `words[i+1]` |
| Context window | `" ".join(words[i-10:i+11])` |
| Collect into a list | `before = []` → `before.append(...)` |
| Count a list | `pd.Series(before).value_counts()` |
| List of dicts → table | `pd.DataFrame(hits)` |
| Save a table | `hits_table.to_csv("energy_hits.csv", index=False)` |
| Paragraphs | `text.split("\n")` |

**Next week:** charts from `energy_hits.csv` with seaborn. **In two weeks:** we send the `context` snippets to a language model.